In [3]:
from dotenv import load_dotenv
import os 

load_dotenv()

OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")[:10]
OPENAI_API_KEY

'sk-proj-qe'

In [6]:
# 필수 import v1.0
from langchain_openai.chat_models.base import ChatOpenAI
from langchain_openai.llms.base import OpenAI
from langchain_core.output_parsers.base import BaseOutputParser
from langchain_core.prompts.prompt import PromptTemplate
from langchain_core.prompts.chat import ChatPromptTemplate

In [7]:
chat = ChatOpenAI()

In [9]:
chat.invoke("호날두와 메시 중 가장 최고의 축구선수 한 명을 고르자면?")

AIMessage(content='이 질문에 대한 답은 주관적이며, 축구 팬들 사이에 논란이 될 수 있습니다. 그러나 호날두와 메시는 둘 다 축구 역사상 가장 우수한 선수들 중 한 명으로 꼽히고 있습니다. 호날두는 강인한 체력과 강력한 슈팅 능력을 가지고 있으며, 메시는 뛰어난 기술과 비주얼스타일로 유명합니다. 둘 다 가장 높은 수준의 경쟁력을 갖고 있으며 각자만의 장점을 보유하고 있습니다. 최고의 축구선수를 고르기는 어려운 결정이지만, 둘 다 현재와 역사적으로 축구계에서 뛰어난 성과를 내고 있는 선수들입니다.', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 258, 'prompt_tokens': 38, 'total_tokens': 296, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-3.5-turbo-0125', 'system_fingerprint': None, 'id': 'chatcmpl-DeblEY6rcbCMU7mw4VFayfnQZoTO8', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--019e1b12-46c6-7730-9ff1-695f58409577-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 38, 'output_tokens': 258, 'to

# temperature(창의력)
- 0.0 ~ 0.3: 결정론 - 요약, 추출, 일관됨, 정확성
- 0.5 ~ 0.7: 균형적 - 적당히 자연스러움, 유연성
- 0.8 ~ 1.0+: 무작위 - 창의적이고 예측 불가능

In [10]:
"""
    같은 프롬프트인데도 결과를 다르게 만들 수 있을까?
"""
None

In [16]:
chat = ChatOpenAI(temperature=1.0)
result = chat.invoke("하늘은 무슨 색이니?")
result.content

'하늘의 색은 일반적으로 푸른색이지만, 날씨나 낮 시간에 따라 다양한 색조로 변할 수 있습니다. 일출이나 일몰 시간에는 주로 오렌지나 붉은색으로 변하고, 흐린 날씨에는 회색이나 그늘색으로 보일 수도 있습니다.'

In [19]:
class NewLineOutputParser(BaseOutputParser):
    # 반드시 parse
    def parse(self, text):
        lines = text.split("\n")
        return [line.lstrip("-123456789. ").strip() for line in lines]

In [20]:
newline_parser = NewLineOutputParser()

In [23]:
newline_parser.parse("""- 1. 햄버거\n- 2. 떡볶이\n- 3. 치킨""")

['햄버거', '떡볶이', '치킨']

In [24]:
template = ChatPromptTemplate.from_messages([
    ("system", """
        리스트를 생성하는 기계입니다.
        요청한 모든 리스트에 개수는 최대 {max_length}개 까지만 목록으로 표시하세요.
        그 이상 초과되는 리스트는 답변하지 마세요.
    """),
    ("human", "{question}")
])

prompt = template.format_messages(
    max_length = 5,
    question = "AI를 잘하려면 어떤 것부터 공부해야해?"
)

# 체인(Chain 생성)
- 랭귀지들을 엮는 체인: 랭체인
- "|": 파이프 연산자로 체인을 만든다.

In [29]:
# list
first_chain = template | chat | NewLineOutputParser()

In [30]:
chain_result = first_chain.invoke({
    "max_length": 5,
    "question": "AI를 잘하려면 어떤 것부터 공부해야해?"
})

In [31]:
chain_result

['데이터 분석 및 통계학',
 '프로그래밍 언어 (파이썬, 자바 등)',
 '기계 학습 및 딥러닝',
 '자연어 처리',
 '컴퓨터 비전',
 '강화 학습',
 '데이터 시각화',
 'AI 윤리 및 규제',
 'AI 모델 해석력',
 '0. 연구 논문 및 최신 기술 동향']

In [32]:
# RunnableSequence
# 이전 단계의 출력이 다음 단계의 입력으로 자동 전달되는 파이프라인 객체
print(type(first_chain))

<class 'langchain_core.runnables.base.RunnableSequence'>


## 1. 템플릿 생성

In [50]:
template = ChatPromptTemplate.from_messages([
    ("system", """
        당신은 세계적인 수준의 여행 가이드입니다.
        사람들이 좋아하는 여행 장소를 많이 알고 있습니다.
        설명없이 지역 명소 이름만 목록으로 {max_length}개 까지 답변하세요.
        목록의 개수가 초과하는 것은 답변하지 마세요.
    """),
    ("human", """
        {place} 여행장소 추천해줘!
    """)
])

In [51]:
second_chain = template | chat

In [53]:
trip_result = second_chain.invoke({
    "max_length": 3,
    "place": "부산"
})

In [54]:
trip_result.content

'1. 해운대 해수욕장\n2. 부평깡통시장\n3. 태종대'

## 실습

In [ ]:
# 저녁 메뉴 추천(양식, 중식, 한식 선택할 수 있도록)
# 체인을 생성 후 결과를 출력

In [62]:
template = ChatPromptTemplate.from_messages([
    ("system", """
        당신은 세계적인 요리사입니다.
        한식, 양식, 중식, 일식 중에서 하나를 선택할 수 있습니다.
        설명없이 음식만 목록으로 {max_length}개 까지 답변하세요.
        목록의 개수가 초과하는 것은 답변하지 마세요.
    """),
    ("human", """
        {menu} 저녁 메뉴를 추천해줘!
    """)
])

In [63]:
food_chain = template | chat

In [64]:
menu_result = food_chain.invoke({
    "max_length": 3,
    "menu": "양식"
})

In [65]:
menu_result.content

'- 파스타 알리올리오\n- 그릴드 스테이크\n- 크림 새우 스튜'

## .stream()
- 결과를 실시간 chunk 단위로 쪼개서 응답한다.

In [ ]:
menu_result = food_chain.stream({
    "max_length": 3,
    "menu": "양식"
})